In [1]:
from pymongo import MongoClient
from datetime import datetime
client = MongoClient("mongodb://localhost:27017/")
db = client["StackOverFlow"]
collections=db['allquestions_with_allanswers']

In [2]:
from bs4 import BeautifulSoup
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
import spacy
import re

nlp = spacy.load("en_core_web_sm")

pos_map = {
    "NOUN": wordnet.NOUN,
    "VERB": wordnet.VERB,
    "ADJ": wordnet.ADJ,
    "ADV": wordnet.ADV
}

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def Extract(text):
    cleantext = []

    soup = BeautifulSoup(str(text), "html.parser")

    # remove code
    for tag in soup.find_all(['code', 'pre']):
        tag.decompose()

    text = soup.get_text(separator=" ")

    # cleaning
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    doc = nlp(text)

    for token in doc:
        if token.text in stop_words or not token.is_alpha:
            continue

        pos = pos_map.get(token.pos_, wordnet.NOUN)
        lemma = lemmatizer.lemmatize(token.text, pos)

        cleantext.append(lemma)

    return " ".join(cleantext)


def extract_code(text):
    soup = BeautifulSoup(str(text), "html.parser")

    codes = [tag.get_text(strip=True) for tag in soup.find_all(['code', 'pre'])]

    return codes



In [3]:
data=[]
for val in collections.find():
    data.append(val)

clean_collection=[]
updated_docs={}
for val in data:
    updated_docs=val.copy()
    time_fields = [
        'protected_date',
        'last_activity_date',
        'creation_date',
        'last_edit_date'
    ]
     
    for field in time_fields:
        if field in updated_docs and updated_docs[field]:
            try:
                updated_docs[field] = datetime.fromtimestamp(
                    updated_docs[field]
                ).strftime("%Y-%m-%d %H:%M:%S")
            except:
                updated_docs[field] = None

    if 'body' in updated_docs:
       raw_body = updated_docs['body']

       updated_docs['body'] = {
        'raw': raw_body,
        'clean_text': Extract(raw_body),
        'code': extract_code(raw_body)
        }
       
    if 'AllAnswers' in updated_docs:
        for answer in updated_docs['AllAnswers']:
            if answer['body']:
                answer_body=answer['body']

                answer['body'] = {
                 'raw': answer_body,
                 'clean_text': Extract(answer_body),
                 'code': extract_code(answer_body)
                   }

    clean_collection.append(updated_docs)
clean_collection[0]

{'_id': ObjectId('69c530dece4825bba74f9f0e'),
 'tags': ['android', 'android-studio', 'kotlin'],
 'owner': {'account_id': 11851631,
  'reputation': 1901,
  'user_id': 8672766,
  'user_type': 'registered',
  'profile_image': 'https://lh3.googleusercontent.com/-BGVVCSx4BtI/AAAAAAAAAAI/AAAAAAAAADI/-Ed-8I-Edcc/s256-rj/photo.jpg',
  'display_name': 'Usman Liaqat',
  'link': 'https://stackoverflow.com/users/8672766/usman-liaqat'},
 'is_answered': True,
 'view_count': 229541,
 'protected_date': '2021-09-25 20:43:57',
 'answer_count': 55,
 'score': 190,
 'last_activity_date': '2024-07-21 10:18:23',
 'creation_date': '2020-05-15 00:15:45',
 'last_edit_date': '2020-05-15 01:38:53',
 'question_id': 61807520,
 'content_license': 'CC BY-SA 4.0',
 'link': 'https://stackoverflow.com/questions/61807520/how-to-fix-error-no-signature-of-method-build-ap86oam3dut3pxce3x49rdtma-androi',
 'title': 'How to fix ERROR: No signature of method: build_ap86oam3dut3pxce3x49rdtma.android()?',
 'body': {'raw': '<p>ERR

In [ ]:
clean_collection

In [4]:
len(clean_collection)

3107